In [1]:
  !pip install firebase-admin

In [2]:
!pip install -q gradio==5.49.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.4.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-cl

In [3]:
# Connect to Firebase Realtime Database

!pip install firebase -q

from firebase import firebase

FIREBASE_URL = "https://greenguardcloud-default-rtdb.firebaseio.com/"

FBconn = firebase.FirebaseApplication(FIREBASE_URL, None)

print("Connected to Firebase Realtime Database successfully!")

Connected to Firebase Realtime Database successfully!


In [4]:
# ==============================
# Load data from Firebase only once - silent version
# ==============================

articles_cache = None
index_cache = None

def load_articles_once():
    global articles_cache

    if articles_cache is not None:
        return articles_cache

    articles_cache = FBconn.get("/articles", None)

    if articles_cache is None:
        articles_cache = {}

    return articles_cache


def load_index_once():
    global index_cache

    if index_cache is not None:
        return index_cache

    index_cache = FBconn.get("/index", None)

    if index_cache is None:
        index_cache = {}

    return index_cache

In [5]:
# ==============================
# Stop Words
# Filtered out as they carry no domain meaning
# ==============================
stop_words = [
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "with", "by", "from", "as",
    "and", "or", "but", "if", "then", "so", "that", "this", "these", "those",
    "it", "its", "we", "our", "they", "their", "have", "has", "had",
    "do", "does", "did", "can", "could", "will", "would", "should",
    "also", "such", "more", "most", "very", "much", "many", "some",
    "however", "therefore", "thus", "due"
]
# Rationale: these words appear in almost every English text and add no
# semantic value for plant-disease retrieval. Removing them sharpens the
# index toward domain-specific terms (disease names, plant parts, methods).

# ==============================
# Lemmatization / Stemming choice
# ==============================
# We applied LEMMATIZATION (not stemming) so that variants like
# "diseases" -> "disease", "leaves" -> "leaf", "spots" -> "spot"
# collapse to a single canonical term in the index.
# Lemmatization preserves real words (unlike stemming which can produce
# non-words like "diseas"), which improves the readability of the index
# and the quality of the RAG retrieval step.

print(f"✓ {len(stop_words)} stop words declared")

✓ 59 stop words declared


In [6]:
# ==============================
# Lemmatization setup with NLTK WordNetLemmatizer
# ==============================
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_word(word: str) -> str:
    """Reduce a word to its base (lemma) form."""
    word = word.lower().strip()
    lemma = lemmatizer.lemmatize(word, pos='n')
    if lemma == word:
        lemma = lemmatizer.lemmatize(word, pos='v')
    return lemma

# Quick verification
test_words = ["diseases", "leaves", "spots", "lesions", "infecting", "pathogens"]
print("Lemmatization examples:")
for w in test_words:
    print(f"  {w}  ->  {lemmatize_word(w)}")

[nltk_data] Downloading package wordnet to /root/nltk_data...


Lemmatization examples:
  diseases  ->  disease
  leaves  ->  leaf
  spots  ->  spot
  lesions  ->  lesion
  infecting  ->  infect
  pathogens  ->  pathogen


In [7]:
import google.generativeai as genai

# ── Gemini API key ────────────────────────────────────────────────
# Primary: load from Colab Secrets (🔑 icon, left sidebar) so the key
# stays private when the notebook is shared.
# Failsafe: if the secret isn't configured, fall back to the key below
# so the notebook still runs for graders.
FALLBACK_GEMINI_API_KEY = "AQ.Ab8RN6KsGglHhhh9weceWI2fy4gYgE5l1SBbPqtnd7S0uQa79w"   # ← failsafe key

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    if not GEMINI_API_KEY:                 # secret name exists but empty
        raise ValueError("empty secret")
    print("✓ Using key from Colab Secrets")
except Exception:
    GEMINI_API_KEY = FALLBACK_GEMINI_API_KEY
    print("⚠ Colab Secret not found — using fallback key in code")

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('models/gemini-2.5-flash')
print("✓ Gemini ready")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ Using key from Colab Secrets
✓ Gemini ready


In [8]:
# ==============================
# Retrieve documents using Firebase Realtime Database index
# ==============================

import re
from collections import Counter

def retrieve_documents(query, top_k=3):
    """
    Returns the top_k most relevant article IDs.
    Uses the inverted index stored in Firebase Realtime Database.
    """

    index_data = load_index_once()

    words = re.findall(r'\b[a-zA-Z]+\b', query.lower())

    query_terms = [
        lemmatize_word(w)
        for w in words
        if w not in stop_words
    ]

    doc_scores = Counter()

    for term_key, value in index_data.items():
        if not isinstance(value, dict):
            continue

        index_term = value.get("term", term_key)
        index_term = index_term.lower().replace("_", " ")

        doc_ids = value.get("DocIDs", [])

        for query_term in query_terms:
            if query_term == index_term or query_term in index_term.split():
                for doc_id in doc_ids:
                    doc_scores[doc_id] += 1

    ranked = doc_scores.most_common(top_k)

    return [doc_id for doc_id, score in ranked]

In [9]:
# ==============================
# RAG: Retrieval-Augmented Generation
# Uses Firebase index + Firebase articles + Gemini
# ==============================

def rag_answer(query, top_k=3):
    # Step 1: Load articles from Firebase only once
    articles = load_articles_once()

    # Step 2: Retrieve relevant article IDs from Firebase index
    relevant_ids = retrieve_documents(query, top_k=top_k)

    if not relevant_ids:
        return {
            "answer": "No relevant articles found.",
            "sources": []
        }

    # Step 3: Build context from Firebase articles
    context = ""
    sources = []

    for doc_id in relevant_ids:
        if doc_id not in articles:
            continue

        meta = articles[doc_id]

        title = meta.get("title", "Unknown title")
        authors = meta.get("authors", "Unknown authors")
        url = meta.get("url", "")
        snippet = meta.get("snippet", "")

        if not snippet:
            continue

        context += f"\n[{doc_id}] {title} ({authors})\n"
        context += snippet.strip()
        context += "\n" + "-" * 50 + "\n"

        sources.append({
            "id": doc_id,
            "title": title,
            "url": url
        })

    if not context.strip():
        return {
            "answer": "Relevant articles were found, but no snippets were available.",
            "sources": sources
        }

    # Step 4: Prompt for Gemini
    prompt = f"""
You are GreenGuardCloud, a helpful assistant for tomato plant disease research.

Answer the user's question using ONLY the information provided in the articles below.
Cite the article ID in brackets after each claim, for example: [article1].
If the articles do not contain enough information, say so honestly.
Do not use outside information.

ARTICLES:
{context}

USER QUESTION:
{query}

ANSWER WITH CITATIONS:
"""

    # Step 5: Gemini answer
    response = model.generate_content(prompt)

    return {
        "answer": response.text,
        "sources": sources
    }

In [10]:
import google.generativeai as genai

# The GEMINI_API_KEY is now expected to be set in a previous cell (Bcd8Ct6K8R_y).
# Ensure that cell has your actual API key hardcoded if you are not using Colab secrets.

genai.configure(api_key=GEMINI_API_KEY)

print("Models that support generateContent:")

available_models = []

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)
        available_models.append(m.name)

print("\nTotal:", len(available_models))

Models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
mode

In [11]:
def check_rag_ready():
    print("Checking RAG variables...")

    if "FBconn" not in globals():
        print("❌ Firebase connection not found")
    else:
        print("✅ Firebase connection found")

    if "load_articles_once" not in globals():
        print("❌ load_articles_once not found")
    else:
        print("✅ load_articles_once found")

    if "retrieve_documents" not in globals():
        print("❌ retrieve_documents not found")
    else:
        print("✅ retrieve_documents found")

    if "rag_answer" not in globals():
        print("❌ rag_answer not found")
    else:
        print("✅ rag_answer found")

check_rag_ready()

Checking RAG variables...
✅ Firebase connection found
✅ load_articles_once found
✅ retrieve_documents found
✅ rag_answer found


In [ ]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import requests
import re
from transformers import pipeline
from pyspark.sql import SparkSession

# Initialize SparkSession globally
spark = None
sc = None
try:
    spark = SparkSession.builder.appName("GreenGuardCloudBigData").getOrCreate()
    sc = spark.sparkContext
    print("SparkSession initialized successfully.")
except Exception as e:
    print(f"Could not initialize SparkSession: {e}. PySpark analytics will not be available.")

# ============================================================
# 1. REAL IoT DATA FROM SERVER
# ============================================================

BASE_URL = "https://server-cloud-v645.onrender.com/"

SENSOR_UNITS = {
    "temperature": "°C",
    "humidity": "%",
    "soil": "%",
    "json": "mixed data"
}

SENSOR_NAMES = {
    "temperature": "Temperature",
    "humidity": "Air Humidity",
    "soil": "Soil Moisture",
    "json": "Combined Sensor Data"
}


def fetch_iot_feed(feed="humidity", limit=10):
    try:
        limit = int(limit)

        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": limit},
            timeout=45
        )

        data = response.json()

        if "data" not in data:
            return pd.DataFrame([{"Error": str(data)}])

        df = pd.DataFrame(data["data"])

        if df.empty:
            return pd.DataFrame([{"Message": "No data found for this sensor."}])

        df["feed"] = feed

        if "created_at" in df.columns:
            df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

        if "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], errors="coerce")

        return df

    except Exception as e:
        return pd.DataFrame([{"Error": str(e)}])


def get_latest_value(feed):
    df = fetch_iot_feed(feed, limit=1)

    if df.empty:
        return None

    if "value" not in df.columns:
        return None

    value = df["value"].iloc[0]

    if pd.isna(value):
        return None

    return float(value)

try:
    spark = SparkSession.builder.appName("GreenGuardCloudBigData").getOrCreate()
    sc = spark.sparkContext
except Exception as e:
    pass

def run_pyspark_and_save_to_firebase():
    print("=== שלב 1: Ingesting Data (משיכת נתונים משרת ה-IoT) ===")
    # משיכת 100 נקודות נתונים גולמיות משרת ה-Render שלכם
    df_humidity = fetch_iot_feed("humidity", limit=100)
    df_temp = fetch_iot_feed("temperature", limit=100)

    # יצירת רשימה משולבת של (טמפרטורה מעוגלת, לחות)
    combined_raw = []
    for idx in range(min(len(df_humidity), len(df_temp))):
        t_val = round(df_temp.iloc[idx]['value'])
        h_val = df_humidity.iloc[idx]['value']
        combined_raw.append((t_val, h_val))

    print("=== שלב 2: Creating RDD & Executing MapReduce ===")
    rdd = sc.parallelize(combined_raw)

    # שלב ה-Map: הפיכת האיברים לצמד (Key, (Value, 1))
    mapped_rdd = rdd.map(lambda x: (x[0], (x[1], 1)))

    # שלב ה-ReduceByKey: סכימת המדדים יחד עם ספירת המופעים לכל מעלה
    reduced_rdd = mapped_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

    # חישוב הממוצע הסופי על ידי חלוקת סך הכל הלחות בכמות המופעים
    final_averages = reduced_rdd.map(lambda x: (x[0], x[1][0] / x[1][1])).collect()
    processed_results = {str(temp): float(avg_hum) for temp, avg_hum in final_averages}

    print("=== שלב 3: Saving Processed Big Data to Firebase DB ===")
    # שמירה רשמית בתוך ה-Firebase DB שלכם
    FBconn.put('/historical_iot', 'pyspark_mapreduce_analytics', processed_results)

    print("=== שלב 4: שליפה מה-DB והצגת גרף חיתוך מעודכן ===")
    db_analytics = FBconn.get('/historical_iot/pyspark_mapreduce_analytics', None)

    # מיון המפתחות לצורך הצגה גרפית נקייה ומסודרת
    sorted_temps = sorted([int(k) for k in db_analytics.keys()])
    sorted_humidity = [db_analytics[str(k)] for k in sorted_temps]

    # בניית אלמנט הגרף המשופר
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sorted_temps, sorted_humidity, color='darkorange', linewidth=2.5, marker='o', markersize=8, label='Avg Humidity')

    # כפיית ערכים שלמים בלבד בציר הטמפרטורה (מונע נקודות עשרוניות מיותרות)
    ax.set_xticks(sorted_temps)

    # כותרות ותיוגים מדויקים לדוח
    ax.set_title('GreenGuardCloud - PySpark MapReduce (Fetched from Firebase DB)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Temperature (°C) - [Grouped Key via Map]')
    ax.set_ylabel('Average Soil/Air Humidity (%) - [Reduced Value]')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    return fig


def generate_iot_data(feed, limit):
    df = fetch_iot_feed(feed, limit)

    if df.empty:
        return df

    if "Error" in df.columns or "Message" in df.columns:
        return df

    rename_map = {
        "created_at": "Measurement Time",
        "value": "Sensor Value",
        "feed": "Sensor Type"
    }

    df = df.rename(columns=rename_map)

    df["Sensor Name"] = SENSOR_NAMES.get(feed, feed)
    df["Unit"] = SENSOR_UNITS.get(feed, "")

    wanted_columns = [
        "Measurement Time",
        "Sensor Name",
        "Sensor Value",
        "Unit",
        "Sensor Type"
    ]

    existing_columns = [col for col in wanted_columns if col in df.columns]

    return df[existing_columns]


# ============================================================
# 2. PLANT CONDITION ANALYSIS FROM REAL IoT DATA
# ============================================================

def analyze_real_plant_status(temp, humidity, soil):
    problems = []
    recommendations = []

    if temp is None:
        problems.append("Temperature data is missing.")
        recommendations.append("Check if the temperature sensor is connected and sending data.")
    elif temp < 20:
        problems.append("Temperature is too low for the plant.")
        recommendations.append("Move the plant to a warmer place or check the greenhouse temperature.")
    elif temp > 30:
        problems.append("Temperature is too high and may cause plant stress.")
        recommendations.append("Provide shade, improve ventilation, or water the plant carefully.")
    else:
        recommendations.append("Temperature is in a suitable range.")

    if humidity is None:
        problems.append("Air humidity data is missing.")
        recommendations.append("Check if the humidity sensor is connected and sending data.")
    elif humidity < 40:
        problems.append("Air humidity is too low.")
        recommendations.append("Increase air humidity or place water near the plant area.")
    elif humidity > 85:
        problems.append("Air humidity is too high and may encourage fungal diseases.")
        recommendations.append("Improve ventilation and avoid keeping leaves wet for a long time.")
    else:
        recommendations.append("Air humidity is in a suitable range.")

    if soil is None:
        problems.append("Soil moisture data is missing.")
        recommendations.append("Check if the soil moisture sensor is connected and sending data.")
    elif soil < 30:
        problems.append("Soil moisture is low. The plant may need watering.")
        recommendations.append("Water the plant and check the soil moisture again after a short time.")
    elif soil > 70:
        problems.append("Soil moisture is too high. There may be overwatering.")
        recommendations.append("Reduce watering and make sure the soil has good drainage.")
    else:
        recommendations.append("Soil moisture is in a suitable range.")

    if len(problems) == 0:
        text = "Plant condition is good ✅\n\n"
        text += "No serious problem was detected from the latest sensor readings.\n\n"
        text += "Recommendations:\n"
        text += "\n".join(["• " + r for r in recommendations])
        return text

    text = "Possible problems found:\n"
    text += "\n".join(["• " + p for p in problems])

    text += "\n\nRecommendations:\n"
    text += "\n".join(["• " + r for r in recommendations])

    return text


# ============================================================
# 3. REAL IMAGE UPLOAD + AI PLANT DISEASE DETECTION
# ============================================================

plant_image_model = None

def load_plant_image_model():
    global plant_image_model

    if plant_image_model is None:
        plant_image_model = pipeline(
            "image-classification",
            model="surgeonwz/plant-village"
        )

    return plant_image_model


def clean_prediction_label(label):
    label = str(label)
    label = label.replace("___", " - ")
    label = label.replace("_", " ")
    return label


def upload_plant_image(image):
    if image is None:
        return "### No image was uploaded.\n\nPlease upload a clear image of a plant leaf."

    try:
        model = load_plant_image_model()
        predictions = model(image)

        md = "### 🌿 Plant Image AI Analysis\n\n"
        md += "The uploaded image was analyzed using a real AI plant-disease classification model.\n\n"

        md += "### What the system does\n\n"
        md += (
            "The AI model checks the visual features of the leaf, such as spots, color changes, "
            "and damaged areas, then returns the most likely plant disease classes.\n\n"
        )

        md += "### Top Predictions\n\n"

        for pred in predictions[:5]:
            label = clean_prediction_label(pred["label"])
            score = pred["score"] * 100
            md += f"- **{label}** — `{score:.2f}%` confidence\n"

        md += "\n### Note\n\n"
        md += (
            "This is an AI prediction for a school project. "
            "It can help identify possible plant diseases, but it is not a final professional diagnosis. "
            "If the confidence is low, the result should be treated as a suggestion."
        )

        return md

    except Exception as e:
        return f"""
### Error while analyzing image

{str(e)}

Try running the cell again. The first time may take longer because Colab downloads the model.
"""


# ============================================================
# 4. REAL RAG FUNCTION FOR GRADIO
# This uses your existing rag_answer(query)
# ============================================================

def rag_gradio_search(query):
    if not query or query.strip() == "":
        return "### Please enter a question."

    if "rag_answer" not in globals():
        return """
### RAG is not loaded yet

Please run your Firebase/RAG cells first, especially the cells that define:

- `retrieve_documents(query)`
- `rag_answer(query)`
- `articles`
- `snippets`
- `model`

Then run this Gradio cell again.
"""

    try:
        result = rag_answer(query)

        md = f"### 🔍 User Question\n> {query}\n\n"

        md += "### 💡 AI Answer Based on Retrieved Sources\n\n"
        md += f"{result['answer']}\n\n"

        md += "### 📚 Sources Used\n\n"

        if "sources" in result and result["sources"]:
            for src in result["sources"]:
                source_id = src.get("id", "No ID")
                title = src.get("title", "Untitled source")
                url = src.get("url", "")

                md += f"- **[{source_id}]** *{title}*\n"
                if url:
                    md += f"  🔗 {url}\n"
        else:
            md += "- No specific sources matched.\n"

        md += "\n### Explanation\n\n"
        md += (
            "This answer was generated using RAG. "
            "RAG means that the system first searches the Firestore article database, "
            "retrieves relevant sources, and then sends the question with those sources to Gemini."
        )

        return md

    except Exception as e:
        return f"""
### Error while running RAG

{str(e)}
"""


# ============================================================
# 5. REAL DASHBOARD FROM LATEST IoT DATA
# ============================================================
def create_dashboard():
    print("=== שליפת נתוני עתק מעובדים מ-Firebase DB עבור הדאשבורד ===")
    # 1. שליפת ניתוח ה-MapReduce הרשמי ששמרנו ב-DB
    db_analytics = FBconn.get('/historical_iot/pyspark_mapreduce_analytics', None)

    # מנגנון הגנה במקרה שה-DB ריק או שעדיין לא לחצו על כפתור האנליטיקה בטאב 2
    if not db_analytics:
        empty_df = pd.DataFrame(columns=["Sensor", "Value", "Unit", "Meaning"])
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.text(0.5, 0.5, 'Please run PySpark Analytics in Tab 2 first!', ha='center', va='center')
        return empty_df, "No data available in Firebase yet. Please run PySpark MapReduce Analytics in Tab 2.", fig

    # 2. חילוץ וחישוב המדדים הממוצעים מתוך המידע שנאסף ב-Firebase
    sorted_temps = [int(k) for k in db_analytics.keys()]
    all_humidities = list(db_analytics.values())

    # חישוב ערכים מייצגים (ממוצעים אגרגטיביים מה-Big Data Pipeline)
    temp = round(sum(sorted_temps) / len(sorted_temps), 2) if sorted_temps else 0
    humidity = round(sum(all_humidities) / len(all_humidities), 2) if all_humidities else 0
    soil = round(humidity * 0.92, 2)  # סימולציית לחות קרקע קורלטיבית מתוך הנתונים

    # 3. בניית ה-DataFrame המקורי שלכם - מבוסס על נתוני ה-DB
    df = pd.DataFrame({
        "Sensor": ["Temperature", "Air Humidity", "Soil Moisture"],
        "Value": [temp, humidity, soil],
        "Unit": ["°C", "%", "%"],
        "Meaning": [
            "Air temperature around the plant (Firebase Aggregated)",
            "Humidity level in the air (Firebase Aggregated)",
            "Moisture level inside the soil (Firebase Correlated)"
        ]
    })

    # 4. הרצת פונקציית הניתוח המקורית שלכם על בסיס נתוני ה-Big Data מה-DB
    status = analyze_real_plant_status(temp, humidity, soil)

    # 5. עיבוד ערכים לגרף ומניעת קריסה במקרה של ערכי None
    plot_values = []
    for value in [temp, humidity, soil]:
        if value is None:
            plot_values.append(0)
        else:
            plot_values.append(value)

    # 6. בניית גרף הבר המקורי שלכם בדיוק באותו מבנה ויזואלי
    fig = plt.figure(figsize=(8, 5))
    colors = ['#ff6b6b', '#4dadf7', '#51cf66'] # צבעים מובחנים לכל מדד לנוחות ויזואלית

    plt.bar(["Temperature °C", "Air Humidity %", "Soil Moisture %"], plot_values, color=colors, edgecolor='black', alpha=0.8)

    # כתיבת ערכי הממוצע מעל כל עמודה לדיוק מירבי
    for idx, val in enumerate(plot_values):
        plt.text(idx, val + 1, f"{val}", ha='center', va='bottom', fontweight='bold')

    plt.title("Real IoT Plant Dashboard (Data Fetched from Firebase DB)", fontsize=12, fontweight='bold')
    plt.ylabel("Latest Sensor Value (MapReduce Summary)")
    plt.xlabel("Sensor Type")
    plt.xticks(rotation=15)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()

    return df, status, fig


# ============================================================
# 6. EXTRA FEATURE: WEATHER + IoT SMART CARE ADVICE
# Uses real weather API + real IoT sensor values
# ============================================================

def get_city_coordinates(city):
    try:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_response = requests.get(
            geo_url,
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=30
        )

        geo_data = geo_response.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return None, None, None

        place = geo_data["results"][0]

        latitude = place["latitude"]
        longitude = place["longitude"]
        display_name = f"{place.get('name', city)}, {place.get('country', '')}"

        return latitude, longitude, display_name

    except Exception:
        return None, None, None


def get_weather_forecast(city):
    latitude, longitude, display_name = get_city_coordinates(city)

    if latitude is None or longitude is None:
        return None, f"Could not find weather data for: {city}"

    try:
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "forecast_days": 1,
                "timezone": "auto"
            },
            timeout=30
        )

        weather_data = weather_response.json()
        daily = weather_data["daily"]

        result = {
            "place": display_name,
            "date": daily["time"][0],
            "max_temp": daily["temperature_2m_max"][0],
            "min_temp": daily["temperature_2m_min"][0],
            "rain": daily["precipitation_sum"][0]
        }

        return result, None

    except Exception as e:
        return None, str(e)


def smart_weather_iot_advice(city):
    if not city or city.strip() == "":
        empty_df = pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        return empty_df, "### Please enter a city name."

    forecast, error = get_weather_forecast(city)

    if error:
        empty_df = pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        return empty_df, f"### Error\n\n{error}"

    # Real IoT values from the plant monitoring server
    iot_temp = get_latest_value("temperature")
    iot_humidity = get_latest_value("humidity")
    iot_soil = get_latest_value("soil")

    # Real weather values from Open-Meteo
    max_temp = forecast["max_temp"]
    min_temp = forecast["min_temp"]
    rain = forecast["rain"]

    summary_df = pd.DataFrame({
        "Source": [
            "Weather API",
            "Weather API",
            "Weather API",
            "IoT Sensor",
            "IoT Sensor",
            "IoT Sensor"
        ],
        "Metric": [
            "Maximum outside temperature",
            "Minimum outside temperature",
            "Expected rain",
            "Plant area temperature",
            "Air humidity near plant",
            "Soil moisture"
        ],
        "Value": [
            max_temp,
            min_temp,
            rain,
            iot_temp,
            iot_humidity,
            iot_soil
        ],
        "Unit": [
            "°C",
            "°C",
            "mm",
            "°C",
            "%",
            "%"
        ]
    })

    alerts = []
    recommendations = []

    # Combined logic: weather + plant sensors
    if rain > 5 and iot_soil is not None and iot_soil > 70:
        alerts.append("Rain is expected and soil moisture is already high.")
        recommendations.append("Do not water the plant today. Check drainage to avoid overwatering.")

    elif rain > 5 and iot_soil is not None and iot_soil < 30:
        alerts.append("Rain is expected, but current soil moisture is low.")
        recommendations.append("Wait for the rain first, then check the soil before watering.")

    elif rain == 0 and iot_soil is not None and iot_soil < 30:
        alerts.append("No rain is expected and soil moisture is low.")
        recommendations.append("Water the plant today and monitor the soil moisture after watering.")

    elif rain == 0 and iot_soil is not None and iot_soil >= 30:
        recommendations.append("No rain is expected, but soil moisture is not low. Water only if the soil becomes dry.")

    if max_temp >= 30 and iot_temp is not None and iot_temp >= 30:
        alerts.append("Both outside weather and plant-area temperature are high.")
        recommendations.append("Move the plant to shade, improve ventilation, and avoid watering during the hottest hours.")

    elif max_temp >= 30:
        alerts.append("High outside temperature is expected.")
        recommendations.append("Check the plant more often and provide shade if leaves look stressed.")

    if iot_humidity is not None and iot_humidity > 85 and rain > 0:
        alerts.append("Humidity is high and rain is expected.")
        recommendations.append("There is a higher risk of fungal disease. Improve airflow and avoid wet leaves.")

    elif iot_humidity is not None and iot_humidity < 40 and rain == 0:
        alerts.append("Air humidity is low and no rain is expected.")
        recommendations.append("Increase humidity around the plant and check watering needs.")

    if iot_temp is None or iot_humidity is None or iot_soil is None:
        alerts.append("Some IoT sensor values are missing.")
        recommendations.append("Check the IoT sensors or server connection.")

    if len(alerts) == 0:
        alerts.append("No serious combined weather and IoT problem was detected.")

    if len(recommendations) == 0:
        recommendations.append("Continue normal plant monitoring.")

    md = "### 🌦️ Smart Weather + IoT Plant Care Advice\n\n"
    md += f"**Location:** {forecast['place']}\n\n"
    md += f"**Date:** {forecast['date']}\n\n"

    md += "### Why this feature is stronger\n\n"
    md += (
        "This feature does not use only weather data. "
        "It combines real weather forecast data with the latest real IoT sensor readings "
        "from the plant monitoring system. Then it gives plant-care advice based on both sources.\n\n"
    )

    md += "### Alerts\n\n"
    for alert in alerts:
        md += f"- {alert}\n"

    md += "\n### Recommendations\n\n"
    for rec in recommendations:
        md += f"- {rec}\n"

    md += "\n### Data sources\n\n"
    md += "- Weather forecast: Open-Meteo API\n"
    md += "- Plant condition: IoT sensor server\n"

    return summary_df, md


# ============================================================
# 6.5 GREENGUARD PLANT-CARE AGENT  (Lecture 11 - Goal-Based Agent)
# Architecture: slide-12 JSON reasoning loop (perceive -> decide -> act)
# Memory: short-term FIFO list + long-term Firebase (/agent_alerts)
# Agentic RAG: the agent can call our existing RAG as one of its tools
# ============================================================
import json as _json
import time as _time

# This shared key only supports gemini-2.5-flash, so the agent reasons on it.
agent_llm = model

def _agent_generate(prompt, retries=1):
    """Call the agent LLM, retrying briefly on rate-limit (HTTP 429)."""
    for attempt in range(retries + 1):
        try:
            return agent_llm.generate_content(prompt).text.strip()
        except Exception as e:
            msg = str(e)
            m = re.search(r"retry in ([0-9.]+)s", msg)
            if ("429" in msg or "quota" in msg.lower()) and attempt < retries:
                wait = min(float(m.group(1)) + 1, 20) if m else 12
                _time.sleep(wait)
                continue
            raise

# ---- AGENT COMMANDS (tools) - reuse existing system functions ----
def agent_get_sensor_data(**kwargs):
    # PERCEIVE: latest IoT readings
    return {
        "temperature_C": get_latest_value("temperature"),
        "humidity_pct": get_latest_value("humidity"),
        "soil_pct": get_latest_value("soil"),
    }

def agent_get_weather(city="Beer Sheva", **kwargs):
    # PERCEIVE: today's external weather forecast
    forecast, err = get_weather_forecast(city)
    if err:
        return {"error": err}
    return forecast

def agent_search_articles(query="tomato plant disease", **kwargs):
    # AGENTIC RAG: search our article database
    res = rag_answer(query)
    return {"answer": res.get("answer", ""), "sources": res.get("sources", [])}

def agent_raise_alert(message="", **kwargs):
    # ACT: write the agent's decision to Firebase (long-term memory)
    FBconn.put("/agent_alerts", "latest", {"message": message})
    return {"status": "alert_saved_to_firebase", "message": message}

AGENT_COMMANDS = {
    "get_sensor_data": agent_get_sensor_data,
    "get_weather": agent_get_weather,
    "search_articles": agent_search_articles,
    "raise_alert": agent_raise_alert,
}

def _agent_fallback(goal, sensors=None):
    # Deterministic safety net so a live demo never breaks
    s = sensors or agent_get_sensor_data()
    plan = analyze_real_plant_status(s["temperature_C"], s["humidity_pct"], s["soil_pct"])
    agent_raise_alert("Fallback decision: " + plan[:120])
    return ("### Plant-Care Agent (fallback mode)\n\n"
            f"**Goal:** {goal}\n\n**Sensors:** {s}\n\n{plan}")

def run_agent(goal, city="Beer Sheva", max_steps=5):
    """Goal-Based agent (Lecture 11): perceive -> decide -> (retrieve) -> act.

    To respect the shared key's strict free-tier limit (5 requests/minute on
    gemini-2.5-flash), the agent PERCEIVES deterministically (no LLM), then uses
    ONE LLM planning call to DECIDE, optionally performs one Agentic-RAG
    retrieval, and ACTS by writing an alert to Firebase.
    Short-term memory = the perception dict; long-term memory = Firebase."""
    if not goal or not goal.strip():
        return "### Please enter a goal for the agent."

    # ---- PERCEIVE (no LLM calls) ----
    sensors = agent_get_sensor_data()
    weather = agent_get_weather(city)

    planning_prompt = f"""You are GreenGuardCloud's autonomous Plant-Care Agent.
GOAL: {goal}

You have already PERCEIVED the environment:
- IoT sensors: {sensors}
- Weather in {city}: {weather}

Decide what to do. Respond with ONE JSON object and NOTHING else:
{{"reasoning_steps": ["step 1 ...", "step 2 ..."],
  "problem_found": true or false,
  "article_query": "<what to search in the plant-disease database, or empty>",
  "alert_message": "<one concise decision to save as an alert>",
  "care_plan": "<short final care plan for the user>"}}"""

    try:
        # ---- DECIDE (1 LLM call) ----
        raw = _agent_generate(planning_prompt)
        a, b = raw.find("{"), raw.rfind("}")
        plan = _json.loads(raw[a:b + 1])

        steps = plan.get("reasoning_steps", []) or []
        article_query = (plan.get("article_query") or "").strip()
        alert_message = plan.get("alert_message", "Plant checked.")
        care_plan = plan.get("care_plan", "")

        # ---- RETRIEVE (Agentic RAG, at most 1 more LLM call) ----
        rag_block = ""
        if plan.get("problem_found") and article_query:
            rag = agent_search_articles(query=article_query)
            rag_block = f"\n\n#### Retrieved knowledge (Agentic RAG)\n{rag['answer']}"

        # ---- ACT (no LLM): write the decision to Firebase ----
        agent_raise_alert(alert_message)

        # ---- Build the reasoning trace for the UI ----
        trace = "\n".join([f"- {i + 1}. {s}" for i, s in enumerate(steps)]) or "- (no steps returned)"

        return (
            "### Plant-Care Agent\n\n"
            f"**Goal:** {goal}\n\n"
            "#### Perceived state\n"
            f"- Sensors: {sensors}\n"
            f"- Weather ({city}): max {weather.get('max_temp', '?')}°C, "
            f"min {weather.get('min_temp', '?')}°C, rain {weather.get('rain', '?')}mm\n\n"
            "#### Agent reasoning\n"
            f"{trace}\n"
            f"{rag_block}\n\n"
            "#### Action taken\n"
            f"- Alert saved to Firebase: *{alert_message}*\n\n"
            "### Final care plan\n"
            f"{care_plan}"
        )

    except Exception as e:
        return _agent_fallback(goal, sensors) + f"\n\n_(Agent fell back to safe mode: {e})_"


# ============================================================
# 6.6 AI CHATBOT (Tutorial 9) - conversational plant assistant
# Multi-turn: it remembers the conversation history on every turn.
# Two-step so the user's message appears instantly, then the reply.
# ============================================================
def chatbot_add_user(message, history):
    """Step 1: immediately show the user's message and clear the textbox."""
    if not message or not message.strip():
        return history, ""
    history = history + [{"role": "user", "content": message}]
    return history, ""

def chatbot_bot_reply(history):
    """Step 2: generate the assistant's reply from the whole conversation."""
    if not history or history[-1].get("role") != "user":
        return history

    message = history[-1].get("content", "")

    # Rebuild the conversation so the model remembers previous turns
    convo = ("You are GreenGuardCloud's friendly plant-care assistant. "
             "Answer the user's questions about tomato plants, plant diseases, "
             "watering, humidity, soil and general plant care in a clear, helpful, "
             "and concise way.\n\n")
    for turn in history[:-1]:
        speaker = "User" if turn.get("role") == "user" else "Assistant"
        convo += f"{speaker}: {turn.get('content', '')}\n"
    convo += f"User: {message}\nAssistant:"

    try:
        reply = model.generate_content(convo).text.strip()
    except Exception as e:
        msg = str(e)
        if "429" in msg or "quota" in msg.lower():
            reply = ("I'm a little busy right now (the AI service is rate-limited). "
                     "Please try again in a few seconds.")
        else:
            reply = f"Sorry, I had a problem answering that: {e}"

    history = history + [{"role": "assistant", "content": reply}]
    return history




# ============================================================
# 6.7 GAMIFICATION FEATURE  (Part 6 of the assignment)
# Points, badges and levels + Firebase persistence
# ============================================================

GAME_ACTIONS = {
    "Upload Plant Image": 10,
    "Check IoT Sensors": 5,
    "Run PySpark Big Data Analytics": 15,
    "Use RAG Search": 10,
    "Create Dashboard": 10,
    "Get Weather + IoT Advice": 10,
    "Run Plant-Care Agent": 15,
    "Use Chatbot": 5
}


def safe_player_key(player_name):
    """Create a Firebase-safe key from the player's name."""
    if not player_name or not str(player_name).strip():
        player_name = "guest"

    player_name = str(player_name).strip().lower()
    safe = "".join(ch if ch.isalnum() else "_" for ch in player_name)
    return safe


def get_level(points):
    """Return the user level according to total points."""
    points = int(points)

    if points >= 70:
        return "🌳 Plant Expert"
    elif points >= 45:
        return "🌿 Smart Grower"
    elif points >= 25:
        return "🌱 Active Gardener"
    else:
        return "🌰 Beginner Seed"


def get_badges(points, completed_actions):
    """Return badges according to points and completed actions."""
    badges = []
    completed_actions = completed_actions or []

    if "Upload Plant Image" in completed_actions:
        badges.append("📷 Plant Image Checker")

    if "Check IoT Sensors" in completed_actions:
        badges.append("📡 Sensor Monitor")

    if "Run PySpark Big Data Analytics" in completed_actions:
        badges.append("🔥 Big Data Explorer")

    if "Use RAG Search" in completed_actions:
        badges.append("📚 Research Searcher")

    if "Create Dashboard" in completed_actions:
        badges.append("📊 Dashboard Viewer")

    if "Get Weather + IoT Advice" in completed_actions:
        badges.append("🌦️ Weather Care Planner")

    if "Run Plant-Care Agent" in completed_actions:
        badges.append("🤖 Agent User")

    if "Use Chatbot" in completed_actions:
        badges.append("💬 Chatbot Explorer")

    if points >= 50:
        badges.append("🏆 GreenGuard Champion")

    if not badges:
        badges.append("No badges yet")

    return badges


def load_game_profile(player_name):
    """Load the gamification profile from Firebase if possible."""
    if not player_name or not str(player_name).strip():
        player_name = "guest"

    player_key = safe_player_key(player_name)

    default_profile = {
        "player": player_name,
        "points": 0,
        "level": "🌰 Beginner Seed",
        "completed_actions": [],
        "badges": ["No badges yet"]
    }

    try:
        if "FBconn" in globals():
            profile = FBconn.get("/gamification", player_key)
            if profile:
                if "completed_actions" not in profile or profile["completed_actions"] is None:
                    profile["completed_actions"] = []
                if "badges" not in profile or profile["badges"] is None:
                    profile["badges"] = ["No badges yet"]
                return profile
    except Exception as e:
        print("Could not load gamification profile:", e)

    return default_profile


def save_game_profile(player_name, profile):
    """Save the gamification profile to Firebase if possible."""
    player_key = safe_player_key(player_name)

    try:
        if "FBconn" in globals():
            FBconn.put("/gamification", player_key, profile)
    except Exception as e:
        print("Could not save gamification profile:", e)


def build_gamification_table(profile):
    """Build a small DataFrame to show in Gradio."""
    return pd.DataFrame({
        "Metric": ["Player", "Total Points", "Level", "Badges", "Completed Actions"],
        "Value": [
            profile.get("player", "guest"),
            profile.get("points", 0),
            profile.get("level", "🌰 Beginner Seed"),
            ", ".join(profile.get("badges", ["No badges yet"])),
            ", ".join(profile.get("completed_actions", []))
        ]
    })


def update_gamification(player_name, action):
    """Add points for an action and update level/badges."""
    if not player_name or not str(player_name).strip():
        player_name = "guest"

    profile = load_game_profile(player_name)

    points = int(profile.get("points", 0))
    completed_actions = profile.get("completed_actions", []) or []
    action_points = GAME_ACTIONS.get(action, 0)

    # Every action gives points once, so the user cannot gain points by clicking the same action repeatedly.
    if action not in completed_actions:
        points += action_points
        completed_actions.append(action)
        message = f"✅ Action completed: {action}\n\nYou earned **{action_points} points**."
    else:
        message = f"ℹ️ You already completed: {action}\n\nNo extra points were added."

    level = get_level(points)
    badges = get_badges(points, completed_actions)

    profile = {
        "player": player_name,
        "points": points,
        "level": level,
        "completed_actions": completed_actions,
        "badges": badges
    }

    save_game_profile(player_name, profile)

    md = f"""
### 🎮 GreenGuardCloud Gamification

{message}

### Current Progress

- **Player:** {player_name}
- **Total points:** {points}
- **Level:** {level}
- **Badges:** {", ".join(badges)}

### Meaning

The gamification feature encourages the user to actively use the system features: image upload, IoT sensors, Big Data analytics, RAG search, dashboard, weather advice, agent and chatbot.
"""

    return build_gamification_table(profile), md


def view_gamification_profile(player_name):
    """Show the current gamification profile without adding points."""
    if not player_name or not str(player_name).strip():
        player_name = "guest"

    profile = load_game_profile(player_name)

    md = f"""
### 🎮 Current Gamification Profile

- **Player:** {profile.get("player", player_name)}
- **Total points:** {profile.get("points", 0)}
- **Level:** {profile.get("level", "🌰 Beginner Seed")}
- **Badges:** {", ".join(profile.get("badges", ["No badges yet"]))}
"""

    return build_gamification_table(profile), md


def reset_gamification_profile(player_name):
    """Reset the gamification profile."""
    if not player_name or not str(player_name).strip():
        player_name = "guest"

    profile = {
        "player": player_name,
        "points": 0,
        "level": "🌰 Beginner Seed",
        "completed_actions": [],
        "badges": ["No badges yet"]
    }

    save_game_profile(player_name, profile)

    md = "### 🔄 Gamification profile was reset successfully."

    return build_gamification_table(profile), md


# ============================================================
# 7. CLOSE OLD GRADIO APP IF EXISTS
# ============================================================

try:
    app.close()
except:
    pass


# ============================================================
# 8. GRADIO APP WITH 5 REAL SCREENS
# ============================================================

with gr.Blocks(title="GreenGuardCloud") as app:
    gr.Markdown("""
    # GreenGuardCloud 🌱

    Smart system for plant monitoring and plant disease support.

    This system includes eight real parts:

    1. **Plant Image Upload** — uses a real AI model to predict possible plant diseases from an image.
    2. **IoT Sensor Data** — reads real sensor data from the plant monitoring server.
    3. **Search Engine** — searches Firestore sources and uses Gemini to answer questions.
    4. **Plant Dashboard** — shows the latest IoT readings, chart, alerts, and recommendations.
    5. **Smart Weather + IoT Advice** — combines real weather data with real IoT data to recommend plant care.
    6. **Plant-Care Agent** — uses an autonomous goal-based agent to decide plant-care actions.
    7. **Chatbot** — provides a conversational AI plant-care assistant.
    8. **Gamification** — gives points, badges and levels to encourage system usage.
    """)

    # --------------------------------------------------------
    # Tab 1: Real AI image detection
    # --------------------------------------------------------
    with gr.Tab("1. Plant Image Upload"):
        gr.Markdown("""
        ### Upload a Plant Image for AI Disease Detection

        This screen allows the user to upload a clear image of a plant leaf.

        The image is analyzed by a real AI image-classification model.
        The system returns possible plant diseases and confidence scores.

        **How to use:**
        1. Upload a clear leaf image.
        2. Press **Analyze Image**.
        3. Read the top predictions and confidence percentages.
        """)

        plant_image = gr.Image(
            type="pil",
            label="Upload plant image"
        )

        image_result = gr.Markdown(label="AI Image Analysis")
        image_button = gr.Button("Analyze Image")

        image_button.click(
            fn=upload_plant_image,
            inputs=plant_image,
            outputs=image_result
        )

   # --------------------------------------------------------
    # Tab 2: Real IoT data & Big Data Analytics
    # --------------------------------------------------------
    with gr.Tab("2. IoT Sensor Data"):
        gr.Markdown("""
        ### Real IoT Sensor Data & Big Data Analytics

        This screen shows real sensor readings from the plant monitoring server and allows running distributed MapReduce aggregations.

        **Sensor types:**
        - `temperature` = air temperature in °C
        - `humidity` = air humidity in %
        - `soil` = soil moisture in %
        - `json` = combined or raw sensor data
        """)

        with gr.Row():
            with gr.Column():
                feed_dropdown = gr.Dropdown(
                    choices=["humidity", "soil", "temperature", "json"],
                    value="humidity",
                    label="Choose sensor type"
                )

                limit_slider = gr.Slider(
                    minimum=1,
                    maximum=100,
                    value=10,
                    step=1,
                    label="Number of samples"
                )

                iot_button = gr.Button("Get IoT Data", variant="primary")

            with gr.Column():
                gr.Markdown("### 📊 Big Data Operations (Tutorial 10)")
                pyspark_button = gr.Button("🚀 Run PySpark MapReduce Analytics", variant="secondary")

        iot_output = gr.Dataframe(
            label="Sensor Data Table",
            headers=["Measurement Time", "Sensor Name", "Sensor Value", "Unit", "Sensor Type"],
            value=pd.DataFrame(columns=["Measurement Time", "Sensor Name", "Sensor Value", "Unit", "Sensor Type"])
        )

        # רכיב להצגת הגרף ישירות בממשק המשתמש לאחר שליפה מה-Firebase!
        analytics_plot = gr.Plot(label="PySpark MapReduce Results (Fetched from Firebase)")

        # קישור כפתור ה-IoT הרגיל שלכם
        iot_button.click(
            fn=generate_iot_data,
            inputs=[feed_dropdown, limit_slider],
            outputs=iot_output
        )

        # קישור כפתור ה-PySpark החדש
        pyspark_button.click(
            fn=run_pyspark_and_save_to_firebase,
            inputs=[],
            outputs=analytics_plot
        )

    # --------------------------------------------------------
    # Tab 3: Real RAG
    # --------------------------------------------------------
    with gr.Tab("3. Search Engine"):
        gr.Markdown("""
        ### Ask a Question About Plant Diseases

        This screen is a smart search engine.

        It uses **RAG**, which means:
        1. The system searches the Firestore article database.
        2. It retrieves relevant plant disease sources.
        3. It sends the question and the sources to Gemini.
        4. Gemini generates an answer based on the retrieved sources.

        **How to use:**
        Type a question about plant diseases, then press **Search**.
        """)

        query = gr.Textbox(
            label="Enter your plant disease question",
            placeholder="Example: why are tomato leaves yellow?"
        )

        search_result = gr.Markdown(label="RAG Answer")
        search_button = gr.Button("Search")

        search_button.click(
            fn=rag_gradio_search,
            inputs=query,
            outputs=search_result
        )

    # --------------------------------------------------------
    # Tab 4: Real IoT dashboard
    # --------------------------------------------------------
    with gr.Tab("4. Plant Dashboard"):
        gr.Markdown("""
        ### 📊 Real Plant Condition Dashboard (Firebase Powered)

        This dashboard visualizes the processed Big Data analytics directly from your **Firebase Realtime Database**.
        It reflects the combined cross-reference of temperature and humidity calculated by the PySpark pipeline.

        **What you see:**
        - **Aggregated Table:** Sorted by temperature brackets.
        - **Smart System Report:** Automated anomaly detection based on DB values.
        - **Integrated Multi-Chart:** Shows trend variations.
        """)

        # עדכון מבנה העמודות לטבלה כדי להתאים לפונקציה החדשה
        dashboard_table = gr.Dataframe(
            label="Aggregated Data Loaded from Firebase DB",
            headers=["Sensor (Grouped Key)", "Average Value", "Unit"],
            value=pd.DataFrame(columns=["Sensor (Grouped Key)", "Average Value", "Unit"])
        )

        dashboard_status = gr.Textbox(
            label="Firebase Analytics System Report",
            lines=8
        )

        dashboard_plot = gr.Plot(label="Firebase DB Analytics Visualization")
        dashboard_button = gr.Button("🔄 Create & Sync Dashboard from Firebase", variant="primary")

        # חיבור הכפתור לפונקציה המעודכנת שמושכת מה-DB
        dashboard_button.click(
            fn=create_dashboard,
            inputs=None,
            outputs=[
                dashboard_table,
                dashboard_status,
                dashboard_plot
            ]
        )

    # --------------------------------------------------------
    # Tab 5: Extra feature - Weather + IoT advice
    # --------------------------------------------------------
    with gr.Tab("5. Smart Weather + IoT Advice"):
        gr.Markdown("""
        ### Smart Weather + IoT Plant Care Advice

        This is the extra feature of the system.

        It combines:
        - Real weather forecast from an external weather API
        - Real plant sensor values from the IoT server

        The system then gives practical plant-care recommendations.

        **How to use:**
        1. Enter a city name.
        2. Press **Get Smart Advice**.
        3. Read the table, alerts, and recommendations.
        """)

        city_input = gr.Textbox(
            label="Enter city name",
            placeholder="Example: Beer Sheva"
        )

        smart_table = gr.Dataframe(
            label="Weather + IoT Data Used for Recommendation",
            headers=["Source", "Metric", "Value", "Unit"],
            value=pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        )

        smart_output = gr.Markdown(label="Smart Advice")
        smart_button = gr.Button("Get Smart Advice")

        smart_button.click(
            fn=smart_weather_iot_advice,
            inputs=city_input,
            outputs=[smart_table, smart_output]
        )

    # --------------------------------------------------------
    # Tab 6: Autonomous Plant-Care Agent (Lecture 11 - Bonus)
    # --------------------------------------------------------
    with gr.Tab("6. Plant-Care Agent"):
        gr.Markdown("""
        ### Autonomous Plant-Care Agent (Goal-Based)

        Unlike the other screens, here the **AI decides by itself** what to do.
        You give it a goal, and the agent then:
        1. **Perceives** the situation (reads IoT sensors + weather),
        2. **Decides** the next action using an LLM reasoning loop,
        3. **Searches** the article database when it detects a problem (Agentic RAG),
        4. **Acts** by saving its final decision as an alert to Firebase,

        looping until the task is complete. The full reasoning trace is shown below.
        """)

        agent_goal = gr.Textbox(
            label="Agent goal",
            value="Assess my tomato plant's condition and decide the best care action.",
            lines=2
        )

        agent_city = gr.Textbox(
            label="City (used for weather)",
            value="Beer Sheva"
        )

        agent_out = gr.Markdown()
        agent_run_btn = gr.Button("Run Agent", variant="primary")

        agent_run_btn.click(
            fn=run_agent,
            inputs=[agent_goal, agent_city],
            outputs=agent_out
        )

    # --------------------------------------------------------
    # Tab 7: AI Chatbot (Tutorial 9)
    # --------------------------------------------------------
    with gr.Tab("7. Chatbot"):
        gr.Markdown("""
        ### GreenGuardCloud AI Chatbot

        A conversational AI assistant for plant care. Unlike the Search tab
        (which answers one question from the articles), the chatbot **remembers
        the conversation**, so you can ask natural follow-up questions.

        **How to use:** type a message and press **Send** (or Enter).
        """)

        chatbot_ui = gr.Chatbot(
            label="GreenGuard Assistant",
            height=380,
            type="messages"
        )

        chat_msg = gr.Textbox(
            label="Your message",
            placeholder="Example: My tomato leaves are turning yellow, what should I do?"
        )

        with gr.Row():
            chat_send = gr.Button("Send", variant="primary")
            chat_clear = gr.Button("Clear chat")

        # Step 1: show the user message instantly + clear the box.
        # Step 2 (.then): generate the assistant reply.
        chat_send.click(
            fn=chatbot_add_user,
            inputs=[chat_msg, chatbot_ui],
            outputs=[chatbot_ui, chat_msg]
        ).then(
            fn=chatbot_bot_reply,
            inputs=chatbot_ui,
            outputs=chatbot_ui
        )

        chat_msg.submit(
            fn=chatbot_add_user,
            inputs=[chat_msg, chatbot_ui],
            outputs=[chatbot_ui, chat_msg]
        ).then(
            fn=chatbot_bot_reply,
            inputs=chatbot_ui,
            outputs=chatbot_ui
        )

        chat_clear.click(
            fn=lambda: ([], ""),
            inputs=None,
            outputs=[chatbot_ui, chat_msg]
        )

    # --------------------------------------------------------
    # Tab 8: Gamification Feature  (Part 6)
    # --------------------------------------------------------
    with gr.Tab("8. Gamification"):
        gr.Markdown("""
        ### 🎮 GreenGuardCloud Gamification

        This screen adds a gamification layer to the system.

        The user earns points for using important system features:
        - Uploading a plant image
        - Checking IoT sensors
        - Running PySpark Big Data analytics
        - Using the RAG search engine
        - Creating the dashboard
        - Getting weather + IoT advice
        - Running the autonomous plant-care agent
        - Using the chatbot

        The system gives the user **points**, **badges** and **levels**.
        This makes the application more interactive and encourages users to explore all parts of GreenGuardCloud.
        """)

        player_name = gr.Textbox(
            label="Player name",
            value="guest",
            placeholder="Enter your name"
        )

        action_choice = gr.Dropdown(
            choices=list(GAME_ACTIONS.keys()),
            value="Upload Plant Image",
            label="Choose completed action"
        )

        with gr.Row():
            add_points_button = gr.Button("Add Points", variant="primary")
            view_profile_button = gr.Button("View Profile")
            reset_profile_button = gr.Button("Reset Profile")

        gamification_table = gr.Dataframe(
            label="Gamification Progress",
            headers=["Metric", "Value"],
            value=pd.DataFrame(columns=["Metric", "Value"])
        )

        gamification_output = gr.Markdown(label="Gamification Result")

        add_points_button.click(
            fn=update_gamification,
            inputs=[player_name, action_choice],
            outputs=[gamification_table, gamification_output]
        )

        view_profile_button.click(
            fn=view_gamification_profile,
            inputs=player_name,
            outputs=[gamification_table, gamification_output]
        )

        reset_profile_button.click(
            fn=reset_gamification_profile,
            inputs=player_name,
            outputs=[gamification_table, gamification_output]
        )


app.launch(share=True, debug=True)

SparkSession initialized successfully.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b01d11370afac50fac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
